In [ ]:
import json
import igraph as ig
import numpy as np
from pathlib import Path
from typing import Dict, List, Any
import time
from datetime import datetime

np.random.seed(42)
ig.random.seed(42)

In [2]:
class FastGraphGenerator:
    """Fast generator for various graph models using igraph"""
    
    def __init__(self, config_path: str, output_dir: str = "graphs"):
        """
        Initialize the graph generator
        
        Args:
            config_path: Path to JSON configuration file
            output_dir: Directory to save generated graphs
        """
        with open(config_path, 'r') as f:
            self.config = json.load(f)
        
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        
        # Create subdirectories for each model
        for model in ['ER', 'BA', 'WS', 'RGG', 'SBM']:
            (self.output_dir / model).mkdir(exist_ok=True)
    
    def _assign_edge_weights(self, num_edges: int, weight_distribution: str = "uniform",
                            weight_params: Dict = None) -> np.ndarray:
        """
        Generate edge weights
        
        Args:
            num_edges: Number of edges
            weight_distribution: Distribution type ('uniform', 'gaussian', 'beta')
            weight_params: Parameters for the distribution
            
        Returns:
            Array of edge weights
        """
        if weight_params is None:
            weight_params = {}
        
        if weight_distribution == "uniform":
            # Uniform distribution in [-1, 1]
            weights = np.random.uniform(-1, 1, num_edges)
        
        elif weight_distribution == "gaussian":
            # Gaussian distribution, clipped to [-1, 1]
            mean = weight_params.get("mean", 0.0)
            std = weight_params.get("std", 0.5)
            weights = np.random.normal(mean, std, num_edges)
            weights = np.clip(weights, -1, 1)
        
        elif weight_distribution == "beta":
            # Beta distribution scaled to [-1, 1]
            alpha = weight_params.get("alpha", 2.0)
            beta = weight_params.get("beta", 2.0)
            weights = np.random.beta(alpha, beta, num_edges)
            weights = weights * 2 - 1  # Scale from [0,1] to [-1,1]
        
        else:
            raise ValueError(f"Unknown weight distribution: {weight_distribution}")
        
        return weights
    
    def _save_graph(self, G: ig.Graph, weights: np.ndarray, filename: str):
        """
        Save graph in the specified format:
        First line: n m (number of nodes and edges)
        Following lines: u v w (edge from u to v with weight w)
        
        Args:
            G: igraph Graph
            weights: Edge weights array
            filename: Output file path
        """
        n = G.vcount()
        m = G.ecount()
        
        with open(filename, 'w') as f:
            # Write header
            f.write(f"{n} {m}\n")
            
            # Get edges as list of tuples
            edges = G.get_edgelist()
            
            # Write edges with weights
            for (u, v), weight in zip(edges, weights):
                f.write(f"{u} {v} {weight:.8f}\n")
    
    def generate_erdos_renyi(self, params: Dict[str, Any], graph_id: int, instance_id: int) -> str:
        """
        Generate Erdős–Rényi random graph
        
        Args:
            params: Dictionary with 'n' (nodes), 'm' (edges),
                   'weight_distribution', and optional 'weight_params'
            graph_id: Unique identifier for this setting
            instance_id: Instance number for this setting
            
        Returns:
            Path to saved graph file
        """
        n = params['n']
        m = params['m']
        weight_dist = params.get('weight_distribution', 'uniform')
        weight_params = params.get('weight_params', {})
        filename = self.output_dir / 'ER' / f"ER_n{n}_m{m}_setting{graph_id}_inst{instance_id}.txt"
        if filename.exists():
            return str(filename)
        
        # Generate graph
        G = ig.Graph.Erdos_Renyi(n=n, m=m, directed=False)
        # Generate edge weights
        weights = self._assign_edge_weights(G.ecount(), weight_dist, weight_params)
        self._save_graph(G, weights, filename)
        
        return str(filename)
    
    def generate_barabasi_albert(self, params: Dict[str, Any], graph_id: int, instance_id: int) -> str:
        """
        Generate Barabási–Albert preferential attachment graph
        
        Args:
            params: Dictionary with 'n' (nodes), 'm' (edges to attach),
                   'weight_distribution', and optional 'weight_params'
            graph_id: Unique identifier for this setting
            instance_id: Instance number for this setting
            
        Returns:
            Path to saved graph file
        """
        n = params['n']
        m = params['m']
        weight_dist = params.get('weight_distribution', 'uniform')
        weight_params = params.get('weight_params', {})
        filename = self.output_dir / 'BA' / f"BA_n{n}_m{m}_setting{graph_id}_inst{instance_id}.txt"
        if filename.exists():
            return str(filename)
        
        # Generate graph
        G = ig.Graph.Barabasi(n=n, m=m, directed=False)
        # Generate edge weights
        weights = self._assign_edge_weights(G.ecount(), weight_dist, weight_params)
        self._save_graph(G, weights, filename)
        
        return str(filename)
    
    def generate_watts_strogatz(self, params: Dict[str, Any], graph_id: int, instance_id: int) -> str:
        """
        Generate Watts–Strogatz small-world graph
        
        Args:
            params: Dictionary with 'n' (nodes), 'k' (nearest neighbors),
                   'p' (rewiring probability), 'weight_distribution',
                   and optional 'weight_params'
            graph_id: Unique identifier for this setting
            instance_id: Instance number for this setting
            
        Returns:
            Path to saved graph file
        """
        n = params['n']
        k = params['k']
        p = params['p']
        weight_dist = params.get('weight_distribution', 'uniform')
        weight_params = params.get('weight_params', {})
        filename = self.output_dir / 'WS' / f"WS_n{n}_k{k}_p{p}_setting{graph_id}_inst{instance_id}.txt"
        if filename.exists():
            return str(filename)
        
        # Generate graph - note: igraph uses 'nei' for number of neighbors (k/2 since it's on each side)
        dim = 1  # 1D lattice
        G = ig.Graph.Watts_Strogatz(dim=dim, size=n, nei=k//2, p=p)
        # Generate edge weights
        weights = self._assign_edge_weights(G.ecount(), weight_dist, weight_params)
        self._save_graph(G, weights, filename)
        
        return str(filename)
    
    def generate_random_geometric(self, params: Dict[str, Any], graph_id: int, instance_id: int) -> str:
        """
        Generate Random Geometric Graph
        
        Args:
            params: Dictionary with 'n' (nodes), 'radius' (connection radius),
                   'weight_distribution', and optional 'weight_params'
            graph_id: Unique identifier for this setting
            instance_id: Instance number for this setting
            
        Returns:
            Path to saved graph file
        """
        n = params['n']
        radius = params['radius']
        weight_dist = params.get('weight_distribution', 'uniform')
        weight_params = params.get('weight_params', {})
        filename = self.output_dir / 'RGG' / f"RGG_n{n}_r{radius}_setting{graph_id}_inst{instance_id}.txt"
        if filename.exists():
            return str(filename)
        
        # Create graph using igraph's GRG (geometric random graph)
        G = ig.Graph.GRG(n, radius)
        # Generate edge weights
        weights = self._assign_edge_weights(G.ecount(), weight_dist, weight_params)
        self._save_graph(G, weights, filename)
        
        return str(filename)
    
    def generate_stochastic_block_model(self, params: Dict[str, Any], graph_id: int, instance_id: int) -> str:
        """
        Generate Stochastic Block Model graph
        
        Args:
            params: Dictionary with 'sizes' (list of block sizes),
                   'p_matrix' (inter/intra-block probabilities),
                   'weight_distribution', and optional 'weight_params'
            graph_id: Unique identifier for this setting
            instance_id: Instance number for this setting
            
        Returns:
            Path to saved graph file
        """
        sizes = params['sizes']
        p_matrix = params['p_matrix']
        weight_dist = params.get('weight_distribution', 'uniform')
        weight_params = params.get('weight_params', {})
        n = sum(sizes)
        filename = self.output_dir / 'SBM' / f"SBM_n{n}_blocks{len(sizes)}_setting{graph_id}_inst{instance_id}.txt"
        if filename.exists():
            return str(filename)
        
        # Generate graph using SBM
        G = ig.Graph.SBM(n, pref_matrix=p_matrix, block_sizes=sizes, directed=False)
        # Generate edge weights
        weights = self._assign_edge_weights(G.ecount(), weight_dist, weight_params)
        self._save_graph(G, weights, filename)
        
        return str(filename)
    
    def generate_all_graphs(self) -> Dict[str, List[str]]:
        """
        Generate all graphs specified in the configuration file
        
        Returns:
            Dictionary mapping model names to lists of generated file paths
        """
        generated_files = {
            'ER': [],
            'BA': [],
            'WS': [],
            'RGG': [],
            'SBM': []
        }
        
        start_time = time.time()
        
        # Calculate total number of graphs
        total_graphs = 0
        for model_configs in self.config.values():
            for config in model_configs:
                num_instances = config.get('num_instances', 5)
                total_graphs += num_instances
        
        current_graph = 0
        
        # Generate Erdős–Rényi graphs
        if 'ER' in self.config:
            for idx, params in enumerate(self.config['ER']):
                num_instances = params.get('num_instances', 5)
                n = params['n']
                m = params['m']
                weight_dist = params.get('weight_distribution', 'uniform')
                
                print(f"Generating ER graphs (setting {idx}): n={n}, m={m}, weight_dist={weight_dist}, instances={num_instances}")
                
                for inst in range(num_instances):
                    filepath = self.generate_erdos_renyi(params, idx, inst)
                    generated_files['ER'].append(filepath)
                    current_graph += 1
                    
                    if current_graph % 10 == 0 or current_graph == total_graphs:
                        self._print_progress(current_graph, total_graphs, start_time)
        
        # Generate Barabási–Albert graphs
        if 'BA' in self.config:
            for idx, params in enumerate(self.config['BA']):
                num_instances = params.get('num_instances', 5)
                n = params['n']
                m = params['m']
                weight_dist = params.get('weight_distribution', 'uniform')
                
                print(f"Generating BA graphs (setting {idx}): n={n}, m={m}, weight_dist={weight_dist}, instances={num_instances}")
                
                for inst in range(num_instances):
                    filepath = self.generate_barabasi_albert(params, idx, inst)
                    generated_files['BA'].append(filepath)
                    current_graph += 1
                    
                    if current_graph % 10 == 0 or current_graph == total_graphs:
                        self._print_progress(current_graph, total_graphs, start_time)
        
        # Generate Watts–Strogatz graphs
        if 'WS' in self.config:
            for idx, params in enumerate(self.config['WS']):
                num_instances = params.get('num_instances', 5)
                n = params['n']
                k = params['k']
                p = params['p']
                weight_dist = params.get('weight_distribution', 'uniform')
                
                print(f"Generating WS graphs (setting {idx}): n={n}, k={k}, p={p}, weight_dist={weight_dist}, instances={num_instances}")
                
                for inst in range(num_instances):
                    filepath = self.generate_watts_strogatz(params, idx, inst)
                    generated_files['WS'].append(filepath)
                    current_graph += 1
                    
                    if current_graph % 10 == 0 or current_graph == total_graphs:
                        self._print_progress(current_graph, total_graphs, start_time)
        
        # Generate Random Geometric graphs
        if 'RGG' in self.config:
            for idx, params in enumerate(self.config['RGG']):
                num_instances = params.get('num_instances', 5)
                n = params['n']
                radius = params['radius']
                weight_dist = params.get('weight_distribution', 'uniform')
                
                print(f"Generating RGG graphs (setting {idx}): n={n}, radius={radius}, weight_dist={weight_dist}, instances={num_instances}")
                
                for inst in range(num_instances):
                    filepath = self.generate_random_geometric(params, idx, inst)
                    generated_files['RGG'].append(filepath)
                    current_graph += 1
                    
                    if current_graph % 10 == 0 or current_graph == total_graphs:
                        self._print_progress(current_graph, total_graphs, start_time)
        
        # Generate Stochastic Block Model graphs
        if 'SBM' in self.config:
            for idx, params in enumerate(self.config['SBM']):
                num_instances = params.get('num_instances', 5)
                n = sum(params['sizes'])
                blocks = len(params['sizes'])
                weight_dist = params.get('weight_distribution', 'uniform')
                
                print(f"Generating SBM graphs (setting {idx}): n={n}, blocks={blocks}, weight_dist={weight_dist}, instances={num_instances}")
                
                for inst in range(num_instances):
                    filepath = self.generate_stochastic_block_model(params, idx, inst)
                    generated_files['SBM'].append(filepath)
                    current_graph += 1
                    
                    if current_graph % 10 == 0 or current_graph == total_graphs:
                        self._print_progress(current_graph, total_graphs, start_time)
        
        elapsed_time = time.time() - start_time
        print(f"\n\nGeneration complete! Total time: {elapsed_time:.2f} seconds")
        print(f"Total graphs generated: {total_graphs}")
        print(f"Average time per graph: {elapsed_time/total_graphs:.3f} seconds")
        
        # Save summary
        self._save_summary(generated_files, elapsed_time)
        
        return generated_files
    
    def _print_progress(self, current: int, total: int, start_time: float):
        """Print progress information"""
        elapsed = time.time() - start_time
        percent = (current / total) * 100
        rate = current / elapsed if elapsed > 0 else 0
        eta = (total - current) / rate if rate > 0 else 0
        print(f"Progress: {current}/{total} ({percent:.1f}%) - Rate: {rate:.2f} graphs/s - ETA: {eta:.1f}s")
    
    def _save_summary(self, generated_files: Dict[str, List[str]], elapsed_time: float):
        """Save a summary of generated graphs"""
        total_graphs = sum(len(files) for files in generated_files.values())
        
        summary = {
            'timestamp': datetime.now().isoformat(),
            'total_graphs': total_graphs,
            'elapsed_time_seconds': elapsed_time,
            'average_time_per_graph': elapsed_time / total_graphs if total_graphs > 0 else 0,
            'models': {
                model: len(files) for model, files in generated_files.items()
            }
        }
        
        summary_path = self.output_dir / 'generation_summary.json'
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2)
        
        print(f"\nSummary saved to: {summary_path}")


def generate_sample_config(output_path: str = "graph_config.json"):
    """
    Generate a sample configuration file with various graph parameters
    
    Args:
        output_path: Path where to save the configuration file
    """
    config = {
        "ER": [],
        "BA": [],
        "WS": [],
        "RGG": [],
        "SBM": []
    }
    
    # Define node counts to test
    node_counts = [10000, 50000, 100000, 500000, 1000000, 5000000, 10000000]
    
    # Weight distributions to use
    weight_distributions = [
        {"weight_distribution": "uniform"},
        {"weight_distribution": "gaussian", "weight_params": {"mean": 0.0, "std": 0.3}},
        {"weight_distribution": "gaussian", "weight_params": {"mean": 0.0, "std": 0.5}},
        {"weight_distribution": "beta", "weight_params": {"alpha": 2.0, "beta": 5.0}},
        {"weight_distribution": "beta", "weight_params": {"alpha": 5.0, "beta": 2.0}}
    ]
    
    # Default number of instances per setting
    default_instances = 1
    
    # Erdős–Rényi: vary edge probability
    for n in node_counts:
        for m in [int(1.1*n), int(1.5*n), int(2.0*n), int(3.0*n), int(5.0*n)]:
            for weight_config in weight_distributions:
                params = {"n": n, "m": m, "num_instances": default_instances}
                params.update(weight_config)
                config["ER"].append(params)
    
    # Barabási–Albert: vary number of edges to attach
    for n in node_counts[:4]:
        for m in [2, 3, 4, 5]:
            for weight_config in weight_distributions:
                params = {"n": n, "m": m, "num_instances": default_instances}
                params.update(weight_config)
                config["BA"].append(params)
    
    # Watts–Strogatz: vary k and rewiring probability
    for n in node_counts[:4]:
        for k in [4, 6, 10]:
            for p in [0.1, 0.3, 0.5]:
                for weight_config in weight_distributions:
                    params = {"n": n, "k": k, "p": p, "num_instances": default_instances}
                    params.update(weight_config)
                    config["WS"].append(params)
    
    # Random Geometric Graph: vary radius
    for n in node_counts[:3]:  # RGG can be memory-intensive
        for radius in [0.01, 0.05]:
            for weight_config in weight_distributions:
                params = {"n": n, "radius": radius, "num_instances": default_instances}
                params.update(weight_config)
                config["RGG"].append(params)
    
    # Stochastic Block Model: vary number of blocks and probabilities
    for n in node_counts[:2]:
        # 2 communities
        sizes_2 = [n // 2, n // 2]
        for p_in in [0.001, 0.005]:
            for p_out in [0.0002, 0.001]:
                for weight_config in weight_distributions:
                    params = {
                        "sizes": sizes_2,
                        "p_matrix": [[p_in, p_out], [p_out, p_in]],
                        "num_instances": default_instances
                    }
                    params.update(weight_config)
                    config["SBM"].append(params)
        # 4 communities
        sizes_4 = [n // 4] * 4
        for p_in in [0.01, 0.05]:
            for p_out in [0.001, 0.005]:
                for weight_config in weight_distributions:
                    params = {
                        "sizes": sizes_4,
                        "p_matrix": [[p_in if i == j else p_out for j in range(4)] for i in range(4)],
                        "num_instances": default_instances
                    }
                    params.update(weight_config)
                    config["SBM"].append(params)
    
    # Save configuration
    with open(output_path, 'w') as f:
        json.dump(config, f, indent=2)
    
    # Calculate total graphs
    total_settings = sum(len(v) for v in config.values())
    total_graphs = 0
    for model_configs in config.values():
        for cfg in model_configs:
            total_graphs += cfg.get('num_instances', default_instances)
    
    print(f"Sample configuration saved to {output_path}")
    print(f"Total settings: {total_settings}")
    print(f"Total graphs to generate: {total_graphs}")
    print(f"  - ER: {len(config['ER'])} settings")
    print(f"  - BA: {len(config['BA'])} settings")
    print(f"  - WS: {len(config['WS'])} settings")
    print(f"  - RGG: {len(config['RGG'])} settings")
    print(f"  - SBM: {len(config['SBM'])} settings")

In [3]:
if __name__ == "__main__":
    # Generate sample configuration file
    print("Generating sample configuration file...")
    generate_sample_config("graph_config.json")
    
    print("\n" + "="*50)
    print("Starting graph generation...")
    print("="*50 + "\n")
    
    # Initialize generator and create graphs
    generator = FastGraphGenerator("graph_config.json", output_dir="../../input/synthetic")
    generated_files = generator.generate_all_graphs()
    
    print("\n" + "="*50)
    print("Graph generation completed!")
    print("="*50)

Generating sample configuration file...
Sample configuration saved to graph_config.json
Total settings: 545
Total graphs to generate: 545
  - ER: 175 settings
  - BA: 80 settings
  - WS: 180 settings
  - RGG: 30 settings
  - SBM: 80 settings

Starting graph generation...

Generating ER graphs (setting 0): n=10000, m=11000, weight_dist=uniform, instances=1
Generating ER graphs (setting 1): n=10000, m=11000, weight_dist=gaussian, instances=1
Generating ER graphs (setting 2): n=10000, m=11000, weight_dist=gaussian, instances=1
Generating ER graphs (setting 3): n=10000, m=11000, weight_dist=beta, instances=1
Generating ER graphs (setting 4): n=10000, m=11000, weight_dist=beta, instances=1
Generating ER graphs (setting 5): n=10000, m=15000, weight_dist=uniform, instances=1
Generating ER graphs (setting 6): n=10000, m=15000, weight_dist=gaussian, instances=1
Generating ER graphs (setting 7): n=10000, m=15000, weight_dist=gaussian, instances=1
Generating ER graphs (setting 8): n=10000, m=1500